# TrustOCT Colab Orchestrator
### Title: TrustOCT: A Trustworthy Retinal OCT Disease Classification Framework Using Domain Generalization and Uncertainty-Aware Selective Prediction

This notebook acts as the high-level Colab controller. All core code is modularized in standalone Python files. The cells in this notebook clone the repository, download datasets, and run training and evaluation wrappers.

### 1. Clone Repository & Install Requirements

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
!git clone https://github.com/Gnanapravallika/TrusthOCTAI.git
%cd TrusthOCTAI
!git pull origin main

# Install dependencies
!pip install -r requirements.txt
!pip install grad-cam

### 2. Dataset Preparation (Kermany OCT2017)

In [ ]:
import os

if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully.')
    except Exception as e:
        print(f'Skipped: {e}')

### 3. Patient-Level Splitting & Path Verification

In [ ]:
from datasets.dataset import auto_detect_columns, patient_level_split

csv_path = 'kermany_dataset_mapping.csv'

if not os.path.exists(csv_path):
    print('CSV not found. Scanning dataset directories...')
    root_oct = None
    for candidate_root in ['/content/Kermany/OCT2017 ', '/content/Kermany/OCT2017', '/content/Kermany', '/content/OCT2017']:
        if os.path.exists(candidate_root):
            root_oct = candidate_root
            break

    if root_oct:
        import pandas as pd
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)
        print(f'Created CSV with {len(df_new)} images.')
    else:
        print('ERROR: No dataset directory found! Please download the Kermany dataset first.')

if os.path.exists(csv_path):
    import pandas as pd
    df = auto_detect_columns(pd.read_csv(csv_path))

    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'

    if os.path.exists('/content') and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        print('Fixing image paths to local Colab storage...')
        def force_local_path(path_str):
            p = path_str.replace('\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath),
                    ]:
                        if os.path.exists(candidate):
                            return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)

    sample_path = df.iloc[0]['image_path']
    print(f'Sample path: {sample_path}')
    print(f'Exists: {os.path.exists(sample_path)}')

    train_df, val_df, test_df = patient_level_split(df)
    print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
else:
    print('ERROR: Dataset CSV not found and could not be generated!')

### 4. Execute Model Training

In [ ]:
#@title Select Active Experiment Settings
EXPERIMENT_NAME = "EXP003_CBAM" #@param ["EXP001_Baseline", "EXP002_MultiScale", "EXP003_CBAM"]

# Copy target experiment model configuration
!cp configs/{EXPERIMENT_NAME}.yaml configs/model.yaml

# Execute training run
!python train.py --experiment_name {EXPERIMENT_NAME}

### 5. Run Evaluation & Generate Explanations

In [ ]:
# Evaluate checkpoints and generate reliability diagrams + LayerCAM heatmaps
!python test.py --experiment_name {EXPERIMENT_NAME}

# Print LayerCAM explainability overlays directly in notebook
from IPython.display import Image, display
overlay_path = f"outputs/{EXPERIMENT_NAME}/explainability_check_layercam.png"
if os.path.exists(overlay_path):
    print("LayerCAM Visual Attribution Overlay:")
    display(Image(filename=overlay_path))

### 6. Single Scan Inference

In [ ]:
#@title Test Prediction on a Sample File
IMAGE_PATH = "datasets/Kermany/OCT2017 /test/NORMAL/NORMAL-1017326-1.jpeg" #@param {type:"string"}

!python inference.py --image_path {IMAGE_PATH} --weights_path outputs/{EXPERIMENT_NAME}/weights_best.pth

### 7. Export Outputs to Google Drive

In [ ]:
# Zip assets and copy to Drive persistently
!zip -r outputs.zip outputs/
!cp outputs.zip "/content/drive/MyDrive/TrustOCT_outputs.zip"
print("[OK] Zipped all checkpoints, metrics, and calibration plots to Drive.")